In [1]:
import os
os.chdir('D:\\vs projects\\yolov7')  



from models.yolo import Model

model = Model(cfg='.//cfg//training//working-final yolov7 mod.yaml')

ModuleNotFoundError: No module named 'mmcv'

In [ ]:
!pip 

In [ ]:
# Mod:
# IDetect(
#     (m): ModuleList(
#       (0): Conv2d(256, 39, kernel_size=(1, 1), stride=(1, 1))
#       (1): Conv2d(512, 39, kernel_size=(1, 1), stride=(1, 1))
#       (2): Conv2d(1024, 39, kernel_size=(1, 1), stride=(1, 1))
#     )
#     (ia): ModuleList(
#       (0-2): 3 x ImplicitA()
#     )
#     (im): ModuleList(
#       (0-2): 3 x ImplicitM()
#     )
#   )O

In [ ]:
# Original:
# IDetect(
#       (m): ModuleList(
#         (0): Conv2d(256, 255, kernel_size=(1, 1), stride=(1, 1))
#         (1): Conv2d(512, 255, kernel_size=(1, 1), stride=(1, 1))
#         (2): Conv2d(1024, 255, kernel_size=(1, 1), stride=(1, 1))
#       )
#       (ia): ModuleList(
#         (0-2): 3 x ImplicitA()
#       )
#       (im): ModuleList(
#         (0-2): 3 x ImplicitM()
#       )
#     )
#   )

In [2]:
yolo_model = Model(cfg='.//cfg//training//yolov7.yaml')

C:\Users\kanna\anaconda3\Lib\site-packages\torch\functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ..\aten\src\ATen\native\TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [3]:
print(yolo_model)

Model(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU()
    )
    (1): Conv(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU()
    )
    (2): Conv(
      (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU()
    )
    (3): Conv(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU()
    )
    (4): Conv(
      (conv): Conv2d(128, 64, kernel_size=(1, 1), 

In [ ]:
# Example of a Deformable Conv layer setup
import torch
import torch.nn as nn

In [ ]:
def autopad(k, p=None):  # kernel, padding
    # Pad to 'same'
    if p is None:
        p = k // 2 if isinstance(k, int) else [x // 2 for x in k]  # auto-pad
    return p

class Conv(nn.Module):
    # Standard convolution
    def __init__(
        self, c1, c2, k=1, s=1, p=None, g=1, act=True
    ):  # ch_in, ch_out, kernel, stride, padding, groups
        super(Conv, self).__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p), groups=g, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = (
            nn.SiLU()
            if act is True
            else (act if isinstance(act, nn.Module) else nn.Identity())
        )

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

    def fuseforward(self, x):
        return self.act(self.conv(x))
    
class DeformConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(DeformConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.offset_conv = nn.Conv2d(in_channels, 2 * kernel_size * kernel_size, kernel_size=3, padding=1)
        
    def forward(self, x):
        offsets = self.offset_conv(x)  # Get the offsets for the deformable convolution
        # Apply the deformable convolution using the calculated offsets
        # This part would include the actual deformable convolution logic
        return self.conv(x)  # Replace this with the actual deformable convolution logic

class MP1(nn.Module):
    def __init__(self, in_channels, kernel_size=2, stride=2):
        super(MP1, self).__init__()
        # Max Pooling with 2x2 kernel and stride of 2 (can be adjusted if necessary)
        self.maxpool = nn.MaxPool2d(kernel_size=kernel_size, stride=stride)
        out_channels = in_channels
        self.branch1_conv1 = Conv(in_channels, out_channels, 1,1)

        self.branch1_conv2 = Conv(in_channels, out_channels, 3,2)

    def forward(self, x):
        x1 = self.maxpool(x)
        x1 = self.branch1_conv1(x1)

        x2 = self.branch1_conv1(x)
        x2 = self.branch1_conv2(x2)
        
        print(x1.shape,x2.shape)
#         # Make sure x2 is downsampled to match x1 size
#         x2 = torch.nn.functional.interpolate(
#             x2, size=x1.shape[2:], mode="nearest"
#         )  # Downsample x2

        return torch.cat([x1, x2], dim=1)
    
class ELAN(nn.Module):
    def __init__(self, in_channels, out_channels, expansion=0.5):
        super(ELAN, self).__init__()
        hidden_channels = int(out_channels * expansion)

        # First branch
        self.branch1_conv1 = Conv(in_channels, hidden_channels, 1, 1)
        self.branch1_conv2 = Conv(hidden_channels, hidden_channels, 3, 1)

        # Second branch
        self.branch2_conv1 = Conv(in_channels, hidden_channels, 1, 1)
        self.branch2_conv2 = Conv(hidden_channels, hidden_channels, 3, 1)

        # Combine outputs
        self.conv_out = Conv(2 * hidden_channels, out_channels, 1, 1)

    def forward(self, x):
        # First branch
        b1 = self.branch1_conv1(x)
        b1 = self.branch1_conv2(b1)

        # Second branch
        b2 = self.branch2_conv1(x)
        b2 = self.branch2_conv2(b2)
            
        # Concatenate both branches and pass through output convolution
        out = torch.cat([b1, b2], dim=1)
        return self.conv_out(out)


In [ ]:
# Sample input tensor
batch_size = 8
in_channels = 256  # Should match the previous layer's output channels
height, width = 32, 32  # Example feature map size
input_tensor = torch.randn(batch_size, in_channels, height, width)

# Create a Deformable Conv layer
dl = DeformConv(in_channels, out_channels=128)
el = ELAN(128,512)
ml = MP1(in_channels=128)

# Forward pass through the Deformable Conv layer
x = dl(input_tensor)
# x=el(x)
op = ml(x)

In [ ]:
op.shape

In [ ]:
input_tensor.shape

In [ ]:
class ELAN(nn.Module):
    
    '''
      Combine CSPNet + VoVNet (OSA Stages)
    '''
    def __init__(self, 
                 n_in, # Input channels
                 n_stage, # Channels in stage
                 n_out, # Output channels
                 n_layers_per_block = 2, # number of layers per block,
                 n_stages_to_combine = 2 # number of stages before aggregating
                ):
        super(ELAN, self).__init__()
        
        # CSP Layers
        self.base_layer = nn.Sequential(
            ConvBnAct(n_in, 
                      2 * n_in, 
                      kernel_size=3, 
                      stride=1, 
                      padding=1, 
                      groups=1, 
                      bn=True, 
                      act=True
                     ), 
            ConvBnAct(2 * n_in, 
                      n_in, 
                      kernel_size=1, 
                      stride=1, 
                      padding=0, 
                      groups=1, 
                      bn=True, 
                      act=True
                     ),
        )
        
        # VoVNet layers
        n_input = n_in//2
        self.stage_layers = []
        
        for _ in range(n_stages_to_combine):
            block_layers = []
            for _ in range(n_layers_per_block):
                block_layers.append(ConvBnAct(n_input, 
                                              n_stage, 
                                              kernel_size=3, 
                                              stride=1, 
                                              padding=1, 
                                              groups=1, 
                                              bn=True, 
                                              act=True
                                             )
                                   ) 
                n_input = n_stage
            
            # Collect feature after every stage
            block_layers = nn.Sequential(*block_layers)
            self.stage_layers.append(block_layers)
        
        self.stage_layers = nn.Sequential(*self.stage_layers)
        
        # Feature Aggregation
        n_input = n_in + n_in//2 + (n_stages_to_combine * n_stage)
        
        self.feat_aggregation = ConvBnAct(n_input, 
                                          n_out, 
                                          kernel_size=1, 
                                          stride=1, 
                                          padding=0, 
                                          groups=1, 
                                          bn=True, 
                                          act=True
                                         )
        
    def forward(self, x):
        
        output = []
        output.append(x)
        
        # Base layer
        x = self.base_layer(x)
        
        # Partition in 2 parts
        x1, x2 = x.chunk(2, dim=1)
        output.append(x1)
        
        for layer in self.stage_layers:
            x2 = layer(x2)
            output.append(x2)
        
        # Concat and Aggregate
        x = torch.cat(output, dim=1)
        x = self.feat_aggregation(x)
        
        return x